In [1]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib import cm

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment
from defdap.quat import Quat
from defdap.plotting import MapPlot

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.linalg import sqrtm, polar
from scipy import stats
from scipy import interpolate 


import skimage as ski


import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [2]:
exp = experiment.Experiment()

# load DIC data 
data_dir = Path('./DIC/pyvale/')
dic_frame = experiment.Frame()

dic_step_list = sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv'))

# for dic_file in sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv')):
#     hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

dic_file = dic_step_list[-2]
hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

hfw = 20.0 # microns
pixelwidth = 2048
pixelsize = hfw/pixelwidth

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_scale(pixelsize)
    dic_map.set_crop(left=100,right=100,top=100,bottom=100)
    # dic_map.plot_map('max_shear',vmin=0,vmax=0.01,plot_scale_bar=True)
    print(dic_map)

Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)


In [3]:
ebsd_frame = experiment.Frame()
data_dir = Path('.')
ebsd.Map(data_dir / 'Pre_EBSD/map.cpr',
         increment=exp.increments[0], frame=ebsd_frame)

Loaded EBSD data (dimensions: 3727 x 2795 pixels, step size: 0.2 um)


In [4]:
ebsd_map = exp.increments[0].maps['ebsd']
# ebsd_map.set_homog_point()

dic_map = exp.increments[0].maps['hrdic']

# dic_map.set_homog_point(vmin=0,vmax=0.05)

In [5]:
ebsd_frame.homog_points = [(1946, 1565),
 (2443, 1000),
 (1305, 1027),
 (1395, 2225),
 (2572, 2193),
 (1876, 1077),
 (2613, 1497),
 (1822, 2208),
 (1259, 1661),
 (2229, 1260),
 (1641, 1325),
 (1693, 1794),
 (2195, 1762)]

In [6]:
dic_frame.homog_points = [(1582, 1380),
 (2615, 172),
 (238, 226),
 (453, 2748),
 (2882, 2728),
 (1431, 326),
 (2970, 1240),
 (1329, 2732),
 (157, 1568),
 (2170, 731),
 (941, 863),
 (1061, 1854),
 (2100, 1795)]

In [7]:
ebsd_map = exp.increments[0].maps['ebsd']

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.link_ebsd_map(ebsd_map, transform_type="polynomial",order=2)
    # dic_map.link_ebsd_map(ebsd_map, transform_type="affine")

In [8]:
dic_map.plot_map('max_shear',vmin=0,vmax=0.1,plot_gbs='pixel',plot_scale_bar=True,dilate_boundaries=True)

Finished building quaternion array (0:00:24) 
Finished finding grain boundaries (0:01:11) 


C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,
C:\Ben\Work\DefDAP\defdap\plotting.py:478: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes

In [9]:
def calc_rotations(dic_map):
    # calculate rotations from dic displacement field 

    # extract deformation gradient
    f = dic_map.data.f

    # calculate rotation as ang = (F21 - F12)/2
    rot = (f[0,1,:,:] - f[1,0,:,:])/2

    # centre on mean 
    rot = rot - np.nanmean(rot)

    return rot 

    


In [13]:
rot = calc_rotations(dic_map)*180/np.pi
plt.figure()
plt.imshow(rot,vmin=-5,vmax=5,cmap='RdBu_r')

In [ ]:
plot = MapPlot.create(dic_map,rot,rot,vmin=-5,vmax=5,cmap='RdBu_r',
                      plot_gbs='pixel',
                      dilate_boundaries=True,
                      boundary_colour='black',
                      plot_colour_bar=True,
                      clabel = 'Rotation / °',
                      plot_scale_bar=True                     
                      )

C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,
C:\Ben\Work\DefDAP\defdap\plotting.py:478: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes

In [18]:
plt.savefig('./figures_for_paper/dic_rotation_max_load_with_gbs.png',dpi=1000)

In [29]:
ebsd_map.plot_map('orientation',component='IPF_x',plot_gbs='pixel',dilate_boundaries=True,plot_scale_bar=True)#,extents=(2000,6000,1000,2500))
ax=plt.gca()
ax.set_xlim([1221,2620])
ax.set_ylim([2327,930])

plt.savefig('./figures_for_paper/ipfx_with_gbs.png',dpi=1000)

In [ ]:
dic_map.plot_map('max_shear',vmin=0,vmax=0.1,plot_gbs='line',plot_scale_bar=True,dilate_boundaries=True)

In [18]:
plt.savefig('./figures_for_paper/max_shear_max_load_with_gbs.png',dpi=1000)

In [46]:
# find special boundaries 
misori_twin = Quat.from_axis_angle([1, 1, 1], 60*np.pi/180)
misori_twin_tol = 10*np.pi/180

# create all symmetric equivalent misorientations
misori_twin_all = []
syms = ebsd_map.primary_phase.crystal_structure.symmetries
for sym_i in syms:
    for sym_j in syms:
        misori_twin_all.append(sym_i.conjugate * misori_twin * sym_j)


# get rid of any duplicates
misori_twin_all = list(set(misori_twin_all))

# calculate neighbour network
ebsd_map.build_neighbour_network()

# loop over all grain boundary segments and check if the misorientation between
# the two grains is within tolerance of the twin misorientation
# store all gbs pairs with misorientation for later us 

all_gb_info = []
twin_gb_info = []
twin_lines = []

for grain1, grain2, b_seg in ebsd_map.neighbour_network.edges.data('boundary'):
    twin = False

    # calculate grain ref orientation
    grain1.calc_average_ori()
    grain2.calc_average_ori()

    misori = grain2.ref_ori * grain1.ref_ori.conjugate

    # # calculate misorientation angle in degrees 
    misori_ang = 2*np.arccos(grain1.ref_ori.mis_ori(grain2.ref_ori,ebsd_map.crystal_sym))*180/np.pi

    # add to list of everything 
    all_gb_info.append([grain1,grain2,misori,misori_ang,b_seg])

    # check if twin and add to twin list
    for misori_twin in misori_twin_all:
        if 2 * np.arccos(misori_twin.dot(misori)) < misori_twin_tol:
            twin = True
            break
    
    if not twin:
        continue
    
    twin_lines.append(b_seg)
    twin_gb_info.append([grain1,grain2,misori,misori_ang,b_seg])




# make into boundary set
s_bounds = ebsd.BoundarySet.from_boundary_segments(twin_lines)

Finished finding grains (0:00:20) twork..
Finished constructing neighbour network (0:00:28) 
